# O2 - fluxos origem → destino (exploração passo a passo)

**O que o enunciado pede.**
1. Contar as viagens de cada mês para todos os pares (`PULocationID`, `DOLocationID`), as "rotas". Juntar a tabela de zonas para ter os nomes e os boroughs da origem e do destino.
2. Calcular, para cada rota, a sua quota no total de viagens que saem da zona de origem nesse mês.
3. Encontrar as 20 rotas com mais viagens em cada borough de origem e detetar as rotas cuja quota mensal **muda de sentido entre o 1º trimestre (Q1) e o 3º (Q3)**.

**Porque é interessante para a otimização.** Há até 263 × 263 ≈ 69 mil rotas por mês, por isso o `groupBy` por rota gera um shuffle bem maior do que os do M2 e do M3. Além disso, a tabela de zonas entra **duas vezes** no join (origem e destino). É aqui que se vê a diferença entre broadcast e sort-merge join e o efeito de agregar antes de juntar.

| Bloco | O quê |
|---|---|
| O2-1 | juntar a tabela de zonas duas vezes (origem e destino) |
| O2-2 | contagem mensal por rota |
| O2-3 | quota da rota nas saídas da zona (window sem ordem) |
| O2-4 | top-20 rotas por borough de origem |
| O2-5 | mudança de sentido Q1 vs Q3 (pivot) |
| O2-6 | resultado, plano e escrita |
| final | tudo junto numa função `run()` |

Carrego **janeiro a setembro de 2019**, porque o passo O2-5 precisa de Q1 (jan–mar) e de Q3 (jul–set). Para explorar mais depressa podes baixar `fraction` (por exemplo para 0.1); as quotas mantêm-se parecidas.

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .master("spark://spark-master:7077")
         .appName("o2-dev")
         .config("spark.driver.memory", "1g")
         .config("spark.executor.memory", "1200m")
         .getOrCreate())

In [ ]:
import sys, contextlib
from types import SimpleNamespace
from pyspark.sql import Window, functions as F
sys.path.insert(0, "/project/src")
import taxi.m1_clean as m1

class NoProf:                      # profiler que não mede nada (o verdadeiro só no main.py)
    def profile(self, name):
        return contextlib.nullcontext()

a = SimpleNamespace(input="/project/data/raw", zones="/project/data/taxi_zone_lookup.csv",
                    output="/project/output", years=[2019], months="1-9", fraction=1.0)
trips, zones = m1.run(spark, a, set(), NoProf())

## Bloco O2-1: tabela de zonas duas vezes (origem e destino)

Cada viagem tem dois números de zona, `PULocationID` (recolha) e `DOLocationID` (largada). Para ter o nome e o borough das duas, a tabela de zonas entra **duas vezes** no join, com nomes de colunas diferentes em cada cópia (`pu_*` e `do_*`). Sem renomear, as duas cópias trariam colunas `Zone` e `Borough` repetidas e o Spark não saberia qual usar (`AMBIGUOUS_REFERENCE`).

**Ordem das operações.** Aqui junta-se primeiro e agrega-se depois, que é o que o M2 e o M3 também fazem. Ou seja, os dois joins são feitos sobre **todas as viagens**. A alternativa é agregar primeiro (no máximo ~69 mil rotas por mês) e juntar depois. É uma das otimizações previstas no `main.py` (`--opt agg_first`) e deve ser medida com o profiler, não assumida.

In [ ]:
pu = zones.select(F.col("LocationID").alias("PULocationID"),
                  F.col("Zone").alias("pu_zone"), F.col("Borough").alias("pu_borough"))
do = zones.select(F.col("LocationID").alias("DOLocationID"),
                  F.col("Zone").alias("do_zone"), F.col("Borough").alias("do_borough"))
t = trips.join(pu, "PULocationID").join(do, "DOLocationID")

t.select("PULocationID", "pu_zone", "pu_borough", "DOLocationID", "do_zone", "do_borough", "month").show(5, truncate=False)

## Bloco O2-2: contagem mensal por rota

Uma linha por (rota, mês) com o número de viagens. As colunas de nome e borough entram no `groupBy` só para as levar até ao resultado: cada ID tem um só nome, por isso não mudam os grupos.

`year` e `month` vêm do nome do ficheiro, como no M1 e no M3.

O resultado (`flows`) é pequeno e vai ser usado em todos os blocos seguintes. Por isso, **só no notebook**, fica em cache: sem cache, cada `show()` voltaria a ler e juntar as viagens todas.

In [ ]:
flows = (t.groupBy("PULocationID", "pu_zone", "pu_borough",
                   "DOLocationID", "do_zone", "do_borough", "year", "month")
          .agg(F.count("*").alias("trip_count")))
flows = flows.cache()              # só no notebook

print("linhas (rota, mês):", flows.count())
flows.orderBy(F.desc("trip_count")).show(5, truncate=False)

## Bloco O2-3: quota da rota nas saídas da zona

```
share = viagens da rota no mês / viagens que saem da zona de origem nesse mês
```

O denominador calcula-se com uma window **sem `orderBy`**:

```python
Window.partitionBy("PULocationID", "year", "month")
```

Sem `orderBy`, a janela é a partição inteira, e por isso `F.sum(...).over(...)` dá o total da zona de origem nesse mês, repetido em cada linha. É o mesmo que fazer um `groupBy` com o total e juntá-lo de volta às rotas, mas sem o join.

O denominador nunca é 0, porque inclui a própria rota, que tem pelo menos 1 viagem. Por isso a divisão normal chega aqui.

In [ ]:
out = Window.partitionBy("PULocationID", "year", "month")
flows = (flows.withColumn("zone_out", F.sum("trip_count").over(out))
              .withColumn("share", F.col("trip_count") / F.col("zone_out")))

# rotas que saem de JFK (132) em janeiro, das mais usadas para as menos
(flows.filter((F.col("PULocationID") == 132) & (F.col("month") == 1))
      .orderBy(F.desc("share"))
      .select("pu_zone", "do_zone", "trip_count", "zone_out", "share").show(5, truncate=False))

**Verificação.** Em cada zona de origem e mês, as quotas de todas as rotas têm de somar **1**.

In [ ]:
(flows.groupBy("PULocationID", "year", "month").agg(F.round(F.sum("share"), 6).alias("soma"))
      .groupBy("soma").count().show())     # deve aparecer só soma = 1.0

## Bloco O2-4: top-20 rotas por borough de origem

1. Total de viagens de cada rota em todo o período carregado: `groupBy` por rota com `sum(trip_count)`.
2. Ranking dentro de cada borough de origem: `Window.partitionBy("pu_borough").orderBy(F.desc("total_trips"))`.
3. Ficar com as posições 1 a 20.

Aqui usamos `row_number` e não `rank`, como no M3: o enunciado pede **20** rotas, e com `rank` um empate na 20ª posição devolveria 21 ou mais. Boroughs com poucas rotas (EWR, por exemplo) podem ter menos de 20.

In [ ]:
routes = (flows.groupBy("PULocationID", "pu_zone", "pu_borough", "DOLocationID", "do_zone", "do_borough")
               .agg(F.sum("trip_count").alias("total_trips")))

by_borough = Window.partitionBy("pu_borough").orderBy(F.desc("total_trips"))
top = (routes.withColumn("route_rank", F.row_number().over(by_borough))
             .filter(F.col("route_rank") <= 20))

top.groupBy("pu_borough").count().show()          # quantas rotas ficaram por borough
(top.filter(F.col("pu_borough") == "Manhattan").orderBy("route_rank")
    .select("route_rank", "pu_zone", "do_zone", "total_trips").show(5, truncate=False))

## Bloco O2-5: a quota muda de sentido entre Q1 e Q3?

**Definição usada.** O "sentido" de um trimestre é se a quota sobe ou desce do 1º para o 3º mês desse trimestre:

```
tendência Q1 = share(março) − share(janeiro)
tendência Q3 = share(setembro) − share(julho)
mudança de sentido = uma é > 0 e a outra < 0   (ou seja, produto < 0)
```

Com 3 meses igualmente espaçados, o declive da reta de regressão é exatamente `(último − primeiro) / 2`. Por isso esta diferença tem sempre o mesmo sinal que uma regressão linear sobre os 3 meses, e o mês do meio não muda o resultado. Uma tendência exatamente 0 conta como "sem sentido" e nunca dá mudança.

**Só para as rotas do top-20.** Em rotas com 2 ou 3 viagens por mês a quota é ruído, e quase todas "mudariam de sentido". Esta escolha deve ficar documentada no relatório.

**Pivot.** Precisamos das quotas de janeiro, março, julho e setembro **na mesma linha**. É isso que o `pivot` faz: transforma valores da coluna `month` em colunas. Passar a lista `[1, 3, 7, 9]` evita uma leitura extra, porque sem ela o Spark teria primeiro de descobrir que valores existem.

**Mês sem viagens.** Se a rota não tiver viagens num desses meses, o pivot deixa `null`. Passamos a 0, que é a quota real nesse mês.

Faz-se por ano. Com vários anos carregados, cada rota tem uma linha por ano.

In [ ]:
q = (flows.join(top.select("PULocationID", "DOLocationID"), ["PULocationID", "DOLocationID"])  # só o top-20
          .filter(F.col("month").isin(1, 3, 7, 9))
          .groupBy("PULocationID", "DOLocationID", "year")
          .pivot("month", [1, 3, 7, 9]).agg(F.first("share"))
          .fillna(0.0, subset=["1", "3", "7", "9"]))

q = (q.select("PULocationID", "DOLocationID", "year",
              F.col("1").alias("share_jan"), F.col("3").alias("share_mar"),
              F.col("7").alias("share_jul"), F.col("9").alias("share_sep"))
      .withColumn("q1_trend", F.col("share_mar") - F.col("share_jan"))
      .withColumn("q3_trend", F.col("share_sep") - F.col("share_jul"))
      .withColumn("share_reversal", F.col("q1_trend") * F.col("q3_trend") < 0))

q.groupBy("share_reversal").count().show()
q.filter("share_reversal").show(5)

## Bloco O2-6: resultado, plano e escrita

Há dois resultados:
- `o2/flows`: todas as rotas por mês, com a contagem e a quota, partido por `pu_borough`;
- `o2/top20`: as top-20 rotas de cada borough, com as quotas de Q1/Q3 e a flag `share_reversal`.

No plano procurar:
- **dois `BroadcastHashJoin`** (ou `SortMergeJoin`) com a tabela de zonas, um para a origem e outro para o destino;
- o **`Exchange`** do `groupBy` por rota, que é o shuffle grande;
- os `Exchange` das windows (`PULocationID, year, month` e `pu_borough`) e o do `pivot`;
- `InMemoryTableScan`: é o `flows` em cache. Aparece várias vezes porque `flows` entra no top-20, no filtro do pivot e no pivot.

In [ ]:
res = (top.join(q, ["PULocationID", "DOLocationID"], "left")
          .orderBy("pu_borough", "route_rank", "year"))
res.explain()

In [ ]:
flows.drop("zone_out").write.mode("overwrite").partitionBy("pu_borough").parquet("/project/output/o2/flows")
res.write.mode("overwrite").parquet("/project/output/o2/top20")
spark.read.parquet("/project/output/o2/top20").show(5, truncate=False)

## Versão final: tudo junto

Os blocos acima numa só função, com a mesma assinatura do M2 e do M3. É isto que iria para `src/taxi/o2_flows.py`, e o `main.py` já sabe chamá-la (`"O2": "o2_flows"`).

**Diferença em relação ao notebook: sem cache.** No pipeline, `flows` não está em cache, e o Spark **não** reutiliza o cálculo entre os vários sítios onde é usado. Confirmei-o no `explain()` com dados de teste: a 2ª escrita lê as viagens 3 vezes (top-20 no resultado, top-20 no filtro do pivot e pivot), e a 1ª escrita mais 1. No total são **4 leituras e joins de todas as viagens**. É a baseline e é um bom primeiro bottleneck a medir: pôr `flows` em cache (`--opt cache`) deve cortar isto para 1 leitura.

In [ ]:
def run(trips, zones, a, opt):
    # 1) tabela de zonas duas vezes: origem (pu_*) e destino (do_*)
    pu = zones.select(F.col("LocationID").alias("PULocationID"),
                      F.col("Zone").alias("pu_zone"), F.col("Borough").alias("pu_borough"))
    do = zones.select(F.col("LocationID").alias("DOLocationID"),
                      F.col("Zone").alias("do_zone"), F.col("Borough").alias("do_borough"))
    t = trips.join(pu, "PULocationID").join(do, "DOLocationID")

    # 2) viagens por (rota, mês) e quota da rota nas saídas da zona de origem
    out = Window.partitionBy("PULocationID", "year", "month")      # sem orderBy: partição inteira
    flows = (t.groupBy("PULocationID", "pu_zone", "pu_borough",
                       "DOLocationID", "do_zone", "do_borough", "year", "month")
              .agg(F.count("*").alias("trip_count"))
              .withColumn("share", F.col("trip_count") / F.sum("trip_count").over(out)))

    # 3) top-20 rotas por borough de origem (row_number: exatamente 20, mesmo com empates)
    by_borough = Window.partitionBy("pu_borough").orderBy(F.desc("total_trips"))
    top = (flows.groupBy("PULocationID", "pu_zone", "pu_borough", "DOLocationID", "do_zone", "do_borough")
                .agg(F.sum("trip_count").alias("total_trips"))
                .withColumn("route_rank", F.row_number().over(by_borough))
                .filter(F.col("route_rank") <= 20))

    # 4) sentido da quota em Q1 (mar - jan) e Q3 (set - jul); mês sem viagens = quota 0
    q = (flows.join(top.select("PULocationID", "DOLocationID"), ["PULocationID", "DOLocationID"])
              .filter(F.col("month").isin(1, 3, 7, 9))
              .groupBy("PULocationID", "DOLocationID", "year")
              .pivot("month", [1, 3, 7, 9]).agg(F.first("share"))      # lista fixa: sem leitura extra
              .fillna(0.0, subset=["1", "3", "7", "9"])
              .select("PULocationID", "DOLocationID", "year",
                      F.col("1").alias("share_jan"), F.col("3").alias("share_mar"),
                      F.col("7").alias("share_jul"), F.col("9").alias("share_sep"))
              .withColumn("q1_trend", F.col("share_mar") - F.col("share_jan"))
              .withColumn("q3_trend", F.col("share_sep") - F.col("share_jul"))
              .withColumn("share_reversal", F.col("q1_trend") * F.col("q3_trend") < 0))
    res = top.join(q, ["PULocationID", "DOLocationID"], "left")

    # 5) escrever: rotas por mês (uma pasta por borough de origem) e top-20 com a flag
    flows.write.mode("overwrite").partitionBy("pu_borough").parquet(f"{a.output}/o2/flows")
    res.write.mode("overwrite").parquet(f"{a.output}/o2/top20")
    return res

final = run(trips, zones, a, set())
final.orderBy("pu_borough", "route_rank").show(5, truncate=False)

In [ ]:
flows.unpersist()